# OmniSub2026 - lipRead Case (All-in-One)

?????? ????????? ???????: ???????? (???????????) + beam inference + wordnorm + ???????? submission.

In [ ]:
from pathlib import Path
import subprocess
import pandas as pd

PROJECT_ROOT = Path('.').resolve()
DATA_ROOT = Path('/kaggle/input/competitions/omni-sub')  # ????????? ??? ????????? ???????
SAMPLE_PATH = DATA_ROOT / 'sample_submission.csv'
CKPT_PATH = PROJECT_ROOT / 'runs/run_char_warmstart_overfit/best.pt'
print('PROJECT_ROOT =', PROJECT_ROOT)
print('DATA_ROOT =', DATA_ROOT)
print('SAMPLE_PATH =', SAMPLE_PATH)
print('CKPT_PATH =', CKPT_PATH)

## 1) (???????????) ???????? from-scratch
?????????? ?????? ???? ?????? ??????????? ??????. ??? ???????? ??????? ???? ??? ????? ??????????.

In [ ]:
# ?????? ??????? ???????? (?????):
# cmd = [
#     'python', 'scripts/quick_ctc_smoke.py',
#     '--data-root', str(DATA_ROOT),
#     '--mode', 'train',
#     '--run-name', 'run_char_warmstart_overfit'
# ]
# subprocess.run(cmd, check=True)
print('Training step is optional and commented out.')

## 2) Beam inference

In [ ]:
beam_csv = PROJECT_ROOT / 'submission_final_beam.csv'
cmd = [
    'python', 'scripts/infer_beam_submission.py',
    '--ckpt', str(CKPT_PATH),
    '--data-root', str(DATA_ROOT),
    '--sample-submission', str(SAMPLE_PATH),
    '--output-csv', str(beam_csv),
    '--beam-size', '30', '--batch-size', '6', '--num-workers', '0'
]
subprocess.run(cmd, check=True)
print('saved:', beam_csv)

## 3) Word normalization

In [ ]:
final_csv = PROJECT_ROOT / 'submission_final_beam_wordnorm.csv'
cmd = [
    'python', 'scripts/apply_wordnorm_from_beam.py',
    '--input-csv', str(PROJECT_ROOT / 'submission_final_beam.csv'),
    '--output-csv', str(final_csv),
]
subprocess.run(cmd, check=True)
print('saved:', final_csv)

## 4) ???????? ??????????

In [ ]:
sub = pd.read_csv(PROJECT_ROOT / 'submission_final_beam_wordnorm.csv')
sample = pd.read_csv(SAMPLE_PATH)
print('rows:', len(sub), 'expected:', len(sample))
print('columns:', list(sub.columns))
print('path_order_match:', sub['path'].astype(str).equals(sample['path'].astype(str)))
sub.head(5)